# Laboratorium 1 - analiza koszykowa

## Przygotowanie

 * pobierz i wypakuj dataset: https://kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba?resource=download&select=basket.csv
   * alternatywnie, pobierz plik `basket.csv` z Teamsów
 * [opcjonalnie] Utwórz wirtualne środowisko
 `python3 -m venv ./recsyslab1`
 * zainstaluj potrzebne biblioteki:
 `pip install more-itertools`

## Część 1. - przygotowanie danych

In [1]:
# importujemy wszystkie potrzebne pakiety

import random
from typing import NamedTuple
from more_itertools import powerset

In [2]:
# definiujemy stałe

PATH = './basket.csv'
EPSILON = 0.001
N = 20          # liczba rekomendacji zwracanych przez recommend()
K = [3, 5, 10]  # wartości k używane przy ewaluacji
N_EVAL = 200    # liczba koszyków użytych do ewaluacji
SEED = 42

In [3]:
# wczytujemy dane o koszykach

def read_baskets(path: str) -> list[set[str]]:
    with open(path) as f:
        raw = f.read()
    baskets = [
        {item.lower() for item in row.split(',') if item}
        for row in raw.split('\n')[1:]
        if row
    ]
    return baskets


def unique_products(baskets: list[set[str]]) -> list[str]:
    products: set[str] = set()
    for basket in baskets:
        products.update(basket)
    return sorted(products)


baskets = read_baskets(PATH)
products = unique_products(baskets)
print(f"Wczytano {len(baskets)} koszyków i {len(products)} unikalnych produktów.")

Wczytano 14963 koszyków i 167 unikalnych produktów.


## Część 2. - obliczanie wskaźników

In [4]:
# obliczamy strukturę danych (słownik) przechowującą wszystkie interesujące wartości `support`
Supports = dict[str, float]

def prods_to_str(prods):
    return '|'.join(prods)

def get_supports(
    baskets: list[set[str]],
    all_products: list[str],
    epsilon: float,
) -> Supports:
    n = len(baskets)
    min_count = epsilon * n

    # create bit mask for every product such that
    # bits are set for every basket containing it
    bitmasks = {p: 0 for p in all_products}
    for i, b in enumerate(baskets):
        for p in b:
            bitmasks[p] |= 1 << i

    supports = {}

    # products set (sorted tuple) -> mask of all buckets containing it
    set_to_mask = {}
    
    # single products
    for p in all_products:
        c = bitmasks[p].bit_count()
        if c > min_count:
            supports[p] = c / n
            set_to_mask[(p,)] = bitmasks[p]

    # keep track of products that are frequent to not try adding infrequent ones
    frequent_products = sorted(p for (p,) in set_to_mask)

    while len(set_to_mask) > 0:
        new_set_to_mask = {}
        # loop over every frequent set and try to add a new product,
        # with every iteration of the while loop cardinality of sets increases:
        # singles -> doubles -> triples, etc.
        for prod_set, mask in set_to_mask.items():
            for p in frequent_products:
                # sets are sorted tuples, so if new product is "earlier" than
                # the last in the set we don't need to check it
                if p <= prod_set[-1]:
                    continue
                new_set = prod_set + (p,)
                new_mask = mask & bitmasks[p]
                c = new_mask.bit_count()
                # any infreqeuent set gets dropped, because extended one
                # would be even more infrequent
                if c > min_count:
                    new_set_to_mask[new_set] = new_mask
                    supports[prods_to_str(new_set)] = c / n
        # new product sets with cardinality increased by 1
        set_to_mask = new_set_to_mask
            
    return supports

supports = get_supports(baskets, products, EPSILON)
supports

{'abrasive cleaner': 0.0014702933903628951,
 'artif. sweetener': 0.0019381140145692708,
 'baking powder': 0.008086613646995923,
 'bathroom cleaner': 0.0011361358016440553,
 'beef': 0.03395041101383412,
 'berries': 0.021787074784468355,
 'beverages': 0.016574216400454454,
 'bottled beer': 0.04531176903027468,
 'bottled water': 0.06068301811134131,
 'brandy': 0.0025395976742631824,
 'brown bread': 0.03762614448974136,
 'butter': 0.03522020985096572,
 'butter milk': 0.017576689166610975,
 'cake bar': 0.006148499632426653,
 'candles': 0.004410880171088686,
 'candy': 0.014368776314910112,
 'canned beer': 0.04691572545612511,
 'canned fish': 0.007685624540533315,
 'canned fruit': 0.001403461872619127,
 'canned vegetables': 0.005480184454988973,
 'cat food': 0.011829178640646929,
 'cereals': 0.002806923745238254,
 'chewing gum': 0.012029673193878232,
 'chicken': 0.027868742899151238,
 'chocolate': 0.02359152576355009,
 'chocolate marshmallow': 0.004009891064626078,
 'citrus fruit': 0.05313105

In [5]:
# definiujemy funkcje obliczające support, confidence i lift

def support(supports: Supports, products: set[str]) -> float:
    prod_str = prods_to_str(sorted(products))
    return supports.get(prod_str, 0.0)


def confidence(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    if support(supports, prior) == 0.0:
        return 0.0
    return support(supports, prior | following) / support(supports, prior)


def lift(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    if support(supports, prior) == 0.0 or support(supports, following) == 0.0:
        return 0.0
    return support(supports, prior | following) / (support(supports, prior) * support(supports, following))

In [6]:
print(support(supports, {'whole milk', 'rolls/buns'}))
print(confidence(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))
print(lift(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))

0.013967787208447505
0.09569377990430622
1.1142926293448512


## Część 3. - generowanie rekomendacji

In [7]:
# wyznaczamy listę potencjalnych rekomendacji
# rekomendowane artykuły powinny mieć lift > 1 i jak najwyższe confidence

class Candidate(NamedTuple):
    item: str
    subbasket: set[str]
    confidence: float
    lift: float


def generate_next_product_candidates(
    basket: set[str],
    products: list[str],
    supports: Supports,
) -> list[Candidate]:
    """Zwraca listę kandydatów posortowanych malejąco po confidence."""
    candidates = []

    for p in products:
        if p in basket:
            continue
        for subbasket in powerset(basket):
            l = lift(supports, set(subbasket), set([p]))
            if l > 1:
                conf = confidence(supports, set(subbasket), set([p]))
                candidates.append(Candidate(p, subbasket, conf, l))

    return sorted(candidates, key=lambda c: c.confidence, reverse=True)

In [8]:
# wybieramy top-n unikalnych produktów z listy kandydatów

def recommend(
    basket: set[str],
    products: list[str],
    supports: Supports,
    n: int,
) -> list[str]:
    """Zwraca top-n rekomendacji na podstawie analizy koszykowej."""
    candidates = generate_next_product_candidates(basket, products, supports)
    res = []
    for c in candidates:
        if c.item not in res:
            res.append(c.item)
            if len(res) >= n:
                break
    return res

In [9]:
print("Koszyk:", baskets[1])
print("Rekomendacje:", recommend(baskets[1], products, supports, N))

Koszyk: {'yogurt', 'whole milk', 'sausage', 'semi-finished bread'}
Rekomendacje: ['rolls/buns', 'soda', 'bottled beer', 'citrus fruit', 'pastry', 'curd', 'frozen vegetables', 'beverages', 'dessert', 'frozen meals', 'salty snack', 'sliced cheese', 'grapes', 'misc. beverages', 'ham', 'chewing gum', 'hard cheese', 'soft cheese', 'specialty bar', 'herbs']


In [10]:
print("Koszyk:", baskets[33])
print("Rekomendacje:", recommend(baskets[33], products, supports, N))

Koszyk: {'photo/film', 'tropical fruit', 'domestic eggs', 'root vegetables', 'soda', 'yogurt', 'white wine'}
Rekomendacje: ['sausage', 'citrus fruit', 'shopping bags', 'newspapers', 'coffee', 'frankfurter', 'frozen vegetables', 'white bread', 'uht-milk', 'specialty chocolate', 'beverages', 'meat', 'waffles', 'oil', 'sliced cheese', 'chewing gum', 'flour', 'grapes', 'processed cheese', 'cat food']


## Część 4. - ewaluacja rekomendacji: MRR@k

### Metryka

**MRR@k** (Mean Reciprocal Rank) mierzy, na jakiej *pozycji* pojawił się ukryty produkt wśród top-k rekomendacji. Nagradza wyższe miejsca na liście:

$$\text{MRR@k} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}_q}$$

gdzie:
- $\mathcal{Q}$ to zbiór przypadków testowych
- $\text{rank}_q$ to pozycja pierwszego trafnego produktu w top-k dla danego przypadku testowego $q$ (lub $\infty$, jeśli nie znaleziono — wkład do sumy wynosi wtedy 0).

### Protokół ewaluacji

Dla każdego koszyka testowego:
1. losowo wydzielamy jeden ukryty produkt; ten produkt byłby tym, który użytkownik najchętniej umieści w koszyku i tym samym tym, który powinien zaproponować nasz algorytm,
2. pozostałe produkty stanowią kontekst (czyli koszyk) przekazywany rekomendatorowi,
3. sprawdzamy, na której pozycji top-k pojawia się ukryty produkt.

Porównamy dwa algorytmy:
- **basket** – rekomendator oparty na analizie koszykowej (confidence + lift),
- **random** – rekomendator losowy (baseline).

Porównanie możemy przeprowadzić dla różnych wartości $k$.

In [11]:
# algorytm losowy - punkt odniesienia (baseline)

def recommend_random(
    basket: set[str],
    products: list[str],
    k: int,
    rng: random.Random,
) -> list[str]:
    """Zwraca k losowo wybranych produktów spoza koszyka."""
    new_products = [p for p in products if p not in basket]
    return rng.sample(new_products, k)

In [12]:
# implementacja metryki MRR@k

def reciprocal_rank_at_k(recommended: list[str], relevant: set[str], k: int) -> float:
    """Zwraca odwrotność pozycji pierwszego trafnego produktu w top-k (0 jeśli brak trafienia)."""
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            return 1 / (i+1)
    return 0.0

In [13]:
# podział koszyka na kontekst i ukryte produkty (ground truth)

def split_basket(
    basket: set[str],
    sample_size: int,
    rng: random.Random,
) -> tuple[set[str], set[str]]:
    """Losowo wydziela `sample_size` produktów jako cel dla rekomendacji; reszta to koszyk"""
    sampled = rng.sample(sorted(basket), sample_size)
    return (set([p for p in basket if p not in sampled]), set(sampled))

In [14]:
# typy reprezentujące wyniki ewaluacji

class MetricAtK(NamedTuple):
    k: int
    mrr: float


class EvaluationResults(NamedTuple):
    basket: MetricAtK
    random: MetricAtK

In [15]:
# ewaluacja obu rekomendatorów dla zadanego k

def evaluate(
    baskets: list[set[str]],
    products: list[str],
    supports: Supports,
    k: int,
    n_eval: int = N_EVAL,
    seed: int = SEED,
) -> EvaluationResults:
    """
    Generuje `n_eval` przypadków testowych przy uzyciu funkcji `split_basket`.
    Dla kazdego przypadku testowego uruchamia oba algorytmy, a następnie oblicza `MMR@k`.
    Zwraca średnie wartości MMR@k dla wskazanego `k` i dla obu algorytmów.
    """
    mmr_random = 0
    mmr_basket = 0
    rng1 = random.Random(seed)
    rng2 = random.Random(seed+1)
    for b in baskets[:n_eval]:
        basket, hidden = split_basket(b, 1, rng1)
        recs_random = recommend_random(basket, products, k, rng2)
        recs_basket = recommend(basket, products, supports, k)
        mmr_random += reciprocal_rank_at_k(recs_random, hidden, k)
        mmr_basket += reciprocal_rank_at_k(recs_basket, hidden, k)
    result_random = MetricAtK(k, mmr_random / n_eval)
    result_basket = MetricAtK(k, mmr_basket / n_eval)
    return EvaluationResults(result_basket, result_random)

In [16]:
# wyświetlamy wyniki i porównujemy algorytmy dla każdego k z listy K

def print_evaluation(results: EvaluationResults) -> None:
    col_w = 10
    header = f"{'Algorytm':<12} | {'mrr@' + str(results.basket.k):>{col_w}}"
    print(header)
    print('-' * len(header))
    for algo, metric in results._asdict().items():
        print(f"{algo:<12} | {metric.mrr:>{col_w}.4f}")


for k in K:
    print_evaluation(evaluate(baskets, products, supports, k))
    print()

Algorytm     |      mrr@3
-------------------------
basket       |     0.0475
random       |     0.0108

Algorytm     |      mrr@5
-------------------------
basket       |     0.0497
random       |     0.0078

Algorytm     |     mrr@10
-------------------------
basket       |     0.0538
random       |     0.0149

